# Step 1 (Colab) — Convert raw logs to a LeRobot dataset

Colab-ready version of `convert_alllog_to_lerobot.ipynb`. Reads raw demos from Google Drive and writes the converted LeRobot dataset back to Drive so it survives runtime resets.

**Expected Drive layout** (adjust `DRIVE_ROOT` below if yours differs):

```
MyDrive/Lebai_train_ACT/Data/log<NNNN>.csv
MyDrive/Lebai_train_ACT/Data/log<NNNN>/ep<NNN>/{color,state,wrist}/...
```

The dataset will land at `MyDrive/Lebai_train_ACT/lerobot_cache/local/lebai_duck_pick/`. Set the same path as `HF_LEROBOT_HOME` in the training notebook so it can find this dataset.

## 1. Install dependencies

In [ ]:
!pip install -q lerobot pandas opencv-python matplotlib tqdm

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 3. Configure paths

Set `HF_LEROBOT_HOME` **before** importing `lerobot` so the dataset lands on Drive.

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Lebai_train_ACT')
LOG_ROOT   = DRIVE_ROOT / 'Data'
LEROBOT_CACHE = DRIVE_ROOT / 'lerobot_cache'
LEROBOT_CACHE.mkdir(parents=True, exist_ok=True)

os.environ['HF_LEROBOT_HOME'] = str(LEROBOT_CACHE)

assert LOG_ROOT.exists(), f'Data folder not found at {LOG_ROOT} — upload Data/ to Drive first.'
print(f'LOG_ROOT       = {LOG_ROOT}')
print(f'HF_LEROBOT_HOME = {os.environ["HF_LEROBOT_HOME"]}')
print(f'Logs found      = {sorted(p.name for p in LOG_ROOT.glob("log[0-9]*.csv"))}')

## 4. Imports

In [ ]:
import json
import shutil

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

try:
    from lerobot.datasets.lerobot_dataset import LeRobotDataset, HF_LEROBOT_HOME
except ImportError:
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset, HF_LEROBOT_HOME
print(f'HF_LEROBOT_HOME resolves to: {HF_LEROBOT_HOME}')

## 5. Conversion configuration

In [ ]:
RUN_NUMBER = None                    # int to pick one log<NNNN>, None to merge all
REPO_ID    = 'local/lebai_duck_pick' # must match DATASET_REPO_ID in the training notebook
INCLUDE_WRIST   = True               # auto-disabled if no wrist data is present
INCLUDE_GRIPPER = True
FPS = 10
IMG_H, IMG_W = 480, 640              # camera service resolution

## 6. Load the index CSV(s)

In [ ]:
def csv_for_run(run_number):
    return LOG_ROOT / f'log{run_number:04d}.csv'

if RUN_NUMBER is not None:
    csv_paths = [csv_for_run(RUN_NUMBER)]
else:
    csv_paths = sorted(LOG_ROOT.glob('log[0-9][0-9][0-9][0-9].csv'))
print(f'CSVs to convert: {[p.name for p in csv_paths]}')

dfs = []
for p in csv_paths:
    d = pd.read_csv(p)
    d['__run_dir__'] = str(LOG_ROOT / p.stem)
    dfs.append(d)
df = pd.concat(dfs, ignore_index=True)
print(f'Total rows: {len(df)}')

In [ ]:
group = df.groupby(['__run_dir__', 'episode'])
summary = group.agg(
    n_frames=('frame', 'count'),
    task=('task', 'first'),
    has_tgt_jp=('tgt_jp0', lambda s: s.notna().any()),
    has_wrist=('wrist', lambda s: s.notna().any() and (s.astype(str).str.len() > 0).any()),
).reset_index()
print(summary.to_string(index=False))

## 7. State and action builders

- `observation.state` = `[jp0..jp5, claw_amplitude]`
- `action` = `[tgt_jp0..tgt_jp5, next_claw_amplitude]` — falls back to `next_row.jp*` when `tgt_jp*` is empty.

In [ ]:
def build_state(row):
    s = [float(row[f'jp{i}']) for i in range(6)]
    if INCLUDE_GRIPPER:
        amp = row.get('claw_amplitude', 0.0)
        s.append(float(amp) if pd.notna(amp) else 0.0)
    return np.array(s, dtype=np.float32)


def build_action(row, next_row):
    tgt = [row.get(f'tgt_jp{i}') for i in range(6)]
    if any(pd.isna(v) for v in tgt):
        tgt = [float(next_row[f'jp{i}']) for i in range(6)]
    else:
        tgt = [float(v) for v in tgt]
    if INCLUDE_GRIPPER:
        amp = next_row.get('claw_amplitude', row.get('claw_amplitude', 0.0))
        tgt.append(float(amp) if pd.notna(amp) else 0.0)
    return np.array(tgt, dtype=np.float32)


def load_rgb(run_dir, rel_path):
    full = Path(run_dir) / rel_path
    bgr = cv2.imread(str(full), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(f'Could not read image: {full}')
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

## 8. Single-frame sanity check

In [ ]:
first_run = sorted(df['__run_dir__'].unique())[0]
ep_df = df[(df['__run_dir__'] == first_run) & (df['episode'] == df['episode'].min())]
ep_df = ep_df.sort_values('frame').reset_index(drop=True)
row = ep_df.iloc[0]
next_row = ep_df.iloc[1] if len(ep_df) > 1 else row

img = load_rgb(row['__run_dir__'], row['color'])
state = build_state(row)
action = build_action(row, next_row)

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(img); ax.axis('off')
ax.set_title(f"task: {row['task']!r}")
plt.show()

print(f'state  ({state.shape}): {np.round(state, 3)}')
print(f'action ({action.shape}): {np.round(action, 3)}')
print(f'state - action delta:   {np.round(state - action, 4)}')

## 9. Create the empty dataset on Drive

In [ ]:
state_dim = 7 if INCLUDE_GRIPPER else 6
action_dim = state_dim
has_wrist_data = INCLUDE_WRIST and df['wrist'].astype(str).str.len().gt(0).any()

features = {
    'observation.images.base': {
        'dtype': 'image', 'shape': (IMG_H, IMG_W, 3),
        'names': ['height', 'width', 'channel'],
    },
    'observation.state': {
        'dtype': 'float32', 'shape': (state_dim,), 'names': ['state'],
    },
    'action': {
        'dtype': 'float32', 'shape': (action_dim,), 'names': ['action'],
    },
}
if has_wrist_data:
    features['observation.images.wrist'] = {
        'dtype': 'image', 'shape': (IMG_H, IMG_W, 3),
        'names': ['height', 'width', 'channel'],
    }

out_path = HF_LEROBOT_HOME / REPO_ID
if out_path.exists():
    print(f'Removing existing dataset at {out_path}')
    shutil.rmtree(out_path)

dataset = LeRobotDataset.create(
    repo_id=REPO_ID,
    robot_type='lebai_lm3',
    fps=FPS,
    features=features,
    image_writer_threads=4,
    image_writer_processes=2,
)
print(f'Created empty dataset at {out_path}')
print(f'  state_dim={state_dim}  action_dim={action_dim}  wrist={has_wrist_data}')

## 10. Conversion loop

Writing JPEGs to Drive is slower than to local SSD. Expect ~minutes per 1000 frames. Don't switch tabs — Colab pauses execution if the page is fully backgrounded.

In [ ]:
total_frames = 0
total_episodes = 0

for (run_dir, ep_idx), ep_df in df.groupby(['__run_dir__', 'episode']):
    ep_df = ep_df.sort_values('frame').reset_index(drop=True)
    task = str(ep_df['task'].iloc[0])
    if not task or task == 'nan':
        print(f'  skipping ep{ep_idx} in {Path(run_dir).name}: empty task'); continue

    n = len(ep_df)
    if n < 5:
        print(f'  skipping ep{ep_idx} in {Path(run_dir).name}: only {n} frames'); continue

    desc = f'{Path(run_dir).name}/ep{ep_idx:03d} [{task[:30]}]'
    for i in tqdm(range(n), desc=desc, leave=False):
        row = ep_df.iloc[i]
        next_row = ep_df.iloc[i + 1] if i + 1 < n else row

        frame = {
            'observation.images.base': load_rgb(row['__run_dir__'], row['color']),
            'observation.state': build_state(row),
            'action': build_action(row, next_row),
            'task': task,
        }
        if has_wrist_data and isinstance(row['wrist'], str) and row['wrist']:
            frame['observation.images.wrist'] = load_rgb(row['__run_dir__'], row['wrist'])

        dataset.add_frame(frame)

    dataset.save_episode()
    total_frames += n
    total_episodes += 1
    print(f'  saved {desc}: {n} frames')

print(f'\nDone. {total_episodes} episodes, {total_frames} frames total.')

## 11. Verify

In [ ]:
ds = LeRobotDataset(REPO_ID)
print(f'Episodes: {ds.num_episodes}')
print(f'Frames:   {ds.num_frames}')
print(f'FPS:      {ds.fps}')
print(f'Features: {list(ds.features.keys())}')

sample = ds[0]
img = sample['observation.images.base']
img_np = img.permute(1, 2, 0).cpu().numpy() if hasattr(img, 'permute') else img
plt.figure(figsize=(6, 4))
plt.imshow(img_np); plt.title(f"task: {sample['task']!r}"); plt.axis('off'); plt.show()
print(f"state:  {sample['observation.state'].numpy().round(3)}")
print(f"action: {sample['action'].numpy().round(3)}")

## Next

Open `act_training_colab.ipynb` — it points at the same `lerobot_cache` directory on Drive.